In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

from jax import jit
from jax.scipy.linalg import expm

In [2]:
from qdots_qll.models.models_scratch_for_drafting import BaseClassDimension

from jaxtyping import Array, Float, Complex, Int, Real

In [3]:
zero = qt.basis(2, 0)
one = qt.basis(2, 1)
plus = (qt.basis(2, 0) + qt.basis(2, 1)).unit()
minus = (qt.basis(2, 0) + 1j * qt.basis(2, 1)).unit()

initial_states = [zero, one, plus, minus]

initial_states_dm = jnp.array(
    [qt.ket2dm(i) for i in initial_states]
)

# canonical_povm = (
#         jnp.array(
#             [
#                 qt.identity(2) + qt.sigmax(),
#                 qt.identity(2) - qt.sigmax(),
#                 qt.identity(2) + qt.sigmay(),
#                 qt.identity(2) - qt.sigmay(),
#                 qt.identity(2) + qt.sigmaz(),
#                 qt.identity(2) - qt.sigmaz(),
#             ]
#         )
#         / 6
# )

povm = (
        jnp.array(
            [
                qt.identity(2) + qt.sigmax(),
                qt.identity(2) - qt.sigmax(),
                qt.identity(2) + qt.sigmay(),
                qt.identity(2) - qt.sigmay(),
                qt.identity(2) + qt.sigmaz(),
                qt.identity(2) - qt.sigmaz(),
            ]
        ) / 2

).reshape(-1, 2, 2, 2)

In [4]:
povm.shape

In [5]:
povm[0]

In [6]:
# Parameters:
big_Omega = 0.5
delta = 0.12739334807998307
eta = np.sqrt(delta ** 2 + big_Omega ** 2)

gamma_minus = 0.15710846160566203
gamma_plus = 0.17916503425352892
S_minus = 0.053851494081252074
S_plus = -0.3336948226536299
S_zero = -delta
T = 30

true_parameters = jnp.array([gamma_minus, gamma_plus, S_minus, S_plus])

In [7]:
G = jnp.array([jnp.array([[1, 0], [0, 1]], ), jnp.array([[0, 1], [1, 0]], ), jnp.array([[0, -1j], [1j, 0]], ),
               jnp.array([[1, 0], [0, -1]], )]) / jnp.sqrt(2)


def rho_to_bloch(rho):
    return jnp.einsum('ijk,kj-> i', G, rho).real


def bloch_to_rho(bloch_v):
    return jnp.einsum('jkl, j', G, bloch_v)


initial_states_bloch = jax.vmap(rho_to_bloch)(initial_states_dm)


class SingleDotWeakCouplingRedfield(BaseClassDimension):
    number_of_parameters: int
    delta: float
    Omega: float
    T: float
    POVM_arr: Complex[Array, "no_basis no_outcomes d d"]
    initial_states_bloch: Float[Array, "no_initial_states d"]
    basis_elements: jax.Array
    trace_povm_G: Float[Array, "no_outcomes d"]

    def __init__(self):
        super().__init__(dimension=2)
        self.number_of_parameters = 4
        self.delta = 0.12739334807998307
        self.Omega = 0.5
        self.T = 30
        self.POVM_arr = povm
        self.basis_elements = jnp.identity(4)
        self.initial_states_bloch = initial_states_bloch
        self.trace_povm_G = jnp.einsum('ijkm,lmk', self.POVM_arr, G).real

    @jit
    def make_bloch_matrix(self, particle):
        delta = self.delta
        big_Omega = self.Omega
        S_zero = -delta

        gamma_minus, gamma_plus, S_minus, S_plus = particle

        big_Gamma = 0.25 * (gamma_plus + gamma_minus)
        small_kappa = 0.25 * (gamma_plus - gamma_minus)
        lamda = 0.5 * (S_plus - S_minus)
        zeta = 0.5 * (S_plus + S_minus)
        delta_prime = delta + S_zero
        big_Omega_prime = big_Omega * (1 + lamda / eta)

        M = jnp.array([
            [0, 0, 0, 0],
            [(- big_Omega * small_kappa) / eta, - (big_Omega / eta) ** 2 * big_Gamma, -delta_prime,
             -(delta * big_Omega) / eta ** 2 * big_Gamma],
            [(delta * big_Omega) / eta ** 2 * (lamda - zeta), delta_prime, - (big_Omega / eta) ** 2 * big_Gamma,
             -big_Omega_prime],
            [0, 0, big_Omega, 0]
        ])
        return M

    # 
    def likelihood_particle(self, particle, t):
        M = self.make_bloch_matrix(particle)
        expMt = expm(M * t)
        evolved_vectors_states = jax.vmap(lambda v0: expMt @ v0)(self.initial_states_bloch)
        p_outcome = jnp.einsum('iz,jkz-> ijk', evolved_vectors_states, self.trace_povm_G)
        # Notation: [init rho, basis, outcome, prob]
        return p_outcome

    def fim(self, particle, t, prob_measurement_basis, prob_initial_state):
        prob_array = self.likelihood_particle(particle, t)
        jac = jax.jacobian(self.likelihood_particle, argnums=0)(particle, t)
        jac = jac.reshape(jac.shape[0], -1)

        prob_over_pbasis_pstate = (
                prob_array / prob_measurement_basis[None, :, None] / prob_initial_state[:, None, None]).flatten()
        fim_element = jax.vmap(lambda x, p: jnp.outer(x, x) / p)(
            jac.T, prob_over_pbasis_pstate
        )
        return fim_element.sum(axis=0)
        # return jnp.where(~jnp.isinf(fim_element), fim_element, 0).sum(axis=0)



# GAME

In [309]:

class SingleDotWeakCouplingGAME(BaseClassDimension):
    number_of_parameters: int
    delta: float
    Omega: float
    T: float
    POVM_arr: Complex[Array, "no_basis no_outcomes d d"]
    initial_states_bloch: Float[Array, "no_initial_states d"]
    basis_elements: jax.Array
    trace_povm_G: Float[Array, "no_outcomes d"]

    def __init__(self):
        super().__init__(dimension=2)
        self.number_of_parameters = 4
        self.delta = 0.12739334807998307
        self.Omega = 0.5
        self.T = 30
        self.POVM_arr = povm
        self.basis_elements = jnp.identity(4)
        self.initial_states_bloch = initial_states_bloch
        self.trace_povm_G = jnp.einsum('ijkm,lmk', self.POVM_arr, G).real

    def make_bloch_matrix(self, particle):
        gn, gp, Sn, Sp = particle
        gnot = 1E-9
        Snot = - self.delta
        # system_hamiltonian = self.delta * jnp.array([[1, 0], [0, -1]]) / 2 + self.Omega * jnp.array(
        #     [[0, 1], [1, 0]]) / 2
        system_hamiltonian = self.delta * jnp.array([[1, 0], [0, 0]]) + self.Omega * jnp.array(
            [[0, 1], [1, 0]]) / 2

        # A = jnp.array([[1, 0], [0, -1]])/2
        A = jnp.array([[1, 0], [0, 0]])

        U = jnp.linalg.eigh(system_hamiltonian)[1]

        Aij = U @ A @ self.dag(U)

        Cp = 0.5 * gp + 1j * Sp
        Cn = 0.5 * gn + 1j * Sn
        Cnot = 0.5 * gnot + 1j * Snot
        Gamma = jnp.array([[Cnot, Cn], [Cp, Cnot]])

        sqrtgamma = jnp.sqrt(jnp.real(Gamma))
        L = jnp.multiply(Aij, sqrtgamma)

        Af = jnp.multiply(Aij, jnp.conjugate(Gamma))

        Hrenorm = -1j / 2 * (Aij @ self.dag(Af) - Af @ self.dag(Aij))

        Htotal = U @ system_hamiltonian @ self.dag(U) + Hrenorm
        liouvillian_energy_basis = (
                -1j * (self.spre(Htotal) - self.spost(Htotal))
                + self.sprepost(self.dag(L), L)
                - 0.5 * (self.spre(L @ self.dag(L)) + self.spost(L @ self.dag(L))))

        matrix_change_basis_bloch = jnp.einsum('kl,ilm,mn,jnk->ij', self.dag(U), G, U, G)

        vec_G = jax.vmap(lambda g: self.vec(g))(G)

        map_bloch_energy_basis = jnp.einsum('ij,jk,lk-> il', jnp.conjugate(vec_G), liouvillian_energy_basis, vec_G)

        map_bloch = (matrix_change_basis_bloch @ map_bloch_energy_basis @ matrix_change_basis_bloch.T)
        return map_bloch

    # 
    # @jit
    # def make_bloch_matrix(self, particle):
    #     delta = self.delta
    #     big_Omega = self.Omega
    #     S_zero = -delta
    # 
    #     gamma_minus, gamma_plus, S_minus, S_plus = particle
    # 
    #     big_Gamma = 0.25 * (gamma_plus + gamma_minus)
    #     small_kappa = 0.25 * (gamma_plus - gamma_minus)
    #     lamda = 0.5 * (S_plus - S_minus)
    #     zeta = 0.5 * (S_plus + S_minus)
    #     delta_prime = delta + S_zero
    #     big_Omega_prime = big_Omega * (1 + lamda / eta)
    # 
    #     M = jnp.array([
    #         [0, 0, 0, 0],
    #         [(- big_Omega * small_kappa) / eta, - (big_Omega / eta) ** 2 * big_Gamma, -delta_prime,
    #          -(delta * big_Omega) / eta ** 2 * big_Gamma],
    #         [(delta * big_Omega) / eta ** 2 * (lamda - zeta), delta_prime, - (big_Omega / eta) ** 2 * big_Gamma,
    #          -big_Omega_prime],
    #         [0, 0, big_Omega, 0]
    #     ])
    #     return M
    # 
    # # 
    def likelihood_particle(self, particle, t):
        M = self.make_bloch_matrix(particle)
        expMt = expm(M * t)
        evolved_vectors_states = jax.vmap(lambda v0: expMt @ v0)(self.initial_states_bloch)
        p_outcome = jnp.einsum('iz,jkz-> ijk', evolved_vectors_states, self.trace_povm_G).real
        # Notation: [init rho, basis, outcome, prob]
        return p_outcome

    def fim(self, particle, t, prob_initial_state, prob_measurement_basis, ):
        prob_array = self.likelihood_particle(particle, t)
        jac = jax.jacobian(self.likelihood_particle, argnums=0)(particle, t)
        jac = jac.reshape(jac.shape[0], -1)

        prob_over_pbasis_pstate = (
                prob_array / prob_measurement_basis[None, :, None] / prob_initial_state[:, None, None]).flatten()
        fim_element = jax.vmap(lambda x, p: jnp.outer(x, x) / p)(
            jac.T, prob_over_pbasis_pstate
        )
        return fim_element.sum(axis=0)
        # return jnp.where(~jnp.isinf(fim_element), fim_element, 0).sum(axis=0)


m = SingleDotWeakCouplingGAME()

m2 = SingleDotWeakCouplingRedfield()

pars1 = true_parameters * jnp.array([2, 2, 1, 1]) / 1
pars2 = true_parameters * jnp.array([1, 1, 1, 1]) / 1

steady_state_sigmax_REDFIELD = -(pars1[1] - pars1[0]) / (pars1[0] + pars1[1]) * eta ** 2 / big_Omega ** 2
steady_state_sigmax_GAME = -(pars2[1] - pars2[0]) / (pars2[0] + pars2[1]) * eta ** 2 / big_Omega ** 2
times = jnp.linspace(0, 120, 500)
evolved_states_probabilities_game = jax.vmap(
    lambda t: m.likelihood_particle(pars1, t))(times)

evolved_states_probabilities_redfield = jax.vmap(
    lambda t: m2.likelihood_particle(pars2, t))(times)
# plt.plot(times, evolved_states_probabilities_old[:, 0, 2, 1])

# plt.plot(times, 2 * evolved_states_probabilities_game[:, 1, 0, 0] - 1, label='Game')
# plt.plot(times, 2 * evolved_states_probabilities_redfield[:, 1, 0, 0] - 1, label='redfield')
# 
# plt.axhline(steady_state_sigmax_GAME, label='GAME')
# plt.axhline(steady_state_sigmax_REDFIELD, label='REDFIELD')

plt.plot(times, evolved_states_probabilities_game[:, 3, 3, 1], label='Game')
plt.plot(times, evolved_states_probabilities_redfield[:, 3, 3, 1], label='redfield')

# plt.yticks(np.arange(-0.08, 0.02,step= 0.02))
plt.legend()
plt.show()

In [277]:
true_parameters

In [198]:

# gn, gp, Sn, Sp = true_parameters
# gnot = 1E-9
# Snot = - delta
# system_hamiltonian = delta * jnp.array([[1, 0], [0, -1]]) / 2 + big_Omega * jnp.array(
#     [[0, 1], [1, 0]]) / 2
# A = jnp.array([[1, 0], [0, -1]])
# 
# U = jnp.linalg.eigh(system_hamiltonian)[1]
# 
# Aij = U @ A @ m.dag(U)
# 
# Cp = 0.5 * gp + 1j * Sp
# Cn = 0.5 * gn + 1j * Sn
# Cnot = 0.5 * gnot + 1j * Snot
# Gamma = jnp.array([[Cnot, Cn], [Cp, Cnot]])
# 
# sqrtgamma = jnp.sqrt(jnp.real(Gamma))
# L = jnp.multiply(Aij, sqrtgamma)
# 
# Af = jnp.multiply(Aij, jnp.conjugate(Gamma))
# 
# Hrenorm = -1j / 2 * (Aij @ m.dag(Af) - Af @ m.dag(Aij))
# 
# Htotal = U @ system_hamiltonian @ m.dag(U) + Hrenorm
# liouvillian_energy_basis = (
#         -1j * (m.spre(Htotal) - m.spost(Htotal))
#         + m.sprepost(m.dag(L), L)
#         - 0.5 * (m.spre(L @ m.dag(L)) + m.spost(L @ m.dag(L))))
# 
# matrix_change_basis_bloch = jnp.einsum('kl,ilm,mn,jnk->ij', m.dag(U), _G, U, _G).real
# 
# # L = m.make_lindbladian(true_parameters)
# vec_G = jax.vmap(lambda g: m.vec(g))(_G)
# 
# map_bloch_energy_basis = jnp.einsum('ij,jk,kl-> il', jnp.conjugate(vec_G), liouvillian_energy_basis, vec_G)
# 
# map_bloch = (matrix_change_basis_bloch @ map_bloch_energy_basis @ matrix_change_basis_bloch.T)
# # return map_bloch


In [296]:
prob_basis = jnp.ones(3) / 3
prob_initial_rho = jnp.ones(4) / 4

In [297]:
jac = jax.jacobian(m.likelihood_particle, 0)(true_parameters, times[20])
jac = jac.reshape(jac.shape[0], -1)
p_array = evolved_states_probabilities[20]
prob_over_pbasis_pstate = (p_array / prob_basis[None, :, None] / prob_initial_rho[:, None, None]).flatten()

fim_element = jax.vmap(lambda x, p: jnp.outer(x, x) / p)(
    jac.T, prob_over_pbasis_pstate
)
jnp.where(~jnp.isinf(fim_element), fim_element, 0).sum(axis=0)

In [298]:
(jax.vmap(lambda t: m.fim(true_parameters, t, prob_basis, prob_initial_rho))(times))[:, 0, 0]

In [310]:
for i in range(4):
    plt.plot(times, jax.vmap(lambda t: m.fim(true_parameters, t, prob_initial_rho, prob_basis, ))(times)[:, i, i],
             label=str(i))

plt.legend()

In [311]:
plt.plot(times, jax.vmap(lambda t: jnp.linalg.det(m.fim(true_parameters, t, prob_initial_rho, prob_basis)))(times))


In [312]:
def det_FIM(particle, t, dist_initial_states, dist_measurements):
    fim = m.fim(particle, t, dist_initial_states, dist_measurements)
    return jnp.linalg.det(fim)

In [313]:
pars = pars1

evolved_jacobians_states = jax.vmap(lambda t: jax.jacobian(det_FIM, 2)(pars, t, prob_initial_rho, prob_basis))(times)
evolved_jacobians_measurements = jax.vmap(lambda t: jax.jacobian(det_FIM, 3)(pars, t, prob_initial_rho, prob_basis))(
    times)



In [314]:
evolved_jacobians_measurements.shape

In [317]:
for i in range(0, 3):
    plt.plot(times, evolved_jacobians_measurements[:, i], label=str(i))

plt.legend()
plt.show()

In [319]:
for i in range(4):
    plt.plot(times, evolved_jacobians_states[:, i], label=str(i))

plt.legend()
plt.show()

In [321]:
evolved_jacobians_states.norm(axis=0)

In [330]:
max_v_initial_state = evolved_jacobians_states[jnp.argmax(jnp.linalg.norm(evolved_jacobians_states, axis=1))]

max_v_measurement_basis = evolved_jacobians_measurements[jnp.argmax(jnp.linalg.norm(evolved_jacobians_measurements, axis=1))]

In [332]:
max_v_initial_state

In [346]:
aux = prob_initial_rho + 0.5*max_v_initial_state
aux = aux/aux.sum()
new_prob_initial_rho = aux
print(new_prob_initial_rho)


aux = prob_basis + 0.5*max_v_measurement_basis
aux = aux/aux.sum()
new_prob_basis = aux
print(new_prob_basis)

In [347]:


plt.plot(times, jax.vmap(lambda t: jnp.linalg.det(m.fim(true_parameters, t, prob_initial_rho, prob_basis)))(times), label='Old')


plt.plot(times, jax.vmap(lambda t: jnp.linalg.det(m.fim(true_parameters, t, new_prob_initial_rho, new_prob_basis)))(times), label='Updated')

plt.legend()
plt.show()